# 03 — Análise em SQL

As mesmas perguntas dos gráficos, respondidas em **SQL** sobre um banco SQLite.

- `src/banco.py` monta `data/ufc.db` a partir dos CSVs limpos, seguindo `sql/schema.sql`: os CSVs (uma linha por lutador em cada luta) viram quatro tabelas — `eventos` → `lutas` → `desempenho` → `desempenho_round`.
- As consultas ficam em `sql/consultas.sql`, cada uma com um `-- nome:`; `src/consultas.py` as lê e executa.
- Cada resposta é **conferida com Pandas**: se os dois caminhos não baterem, a célula falha.

In [1]:
import sys
sys.path.append("..")

import sqlite3
import pandas as pd

from src.banco import criar_banco, ARQUIVO_BANCO
from src.consultas import carregar_consultas, consultar

# Recria o banco a partir dos CSVs (rápido; garante que está em dia com os dados)
for tabela, linhas in criar_banco().items():
    print(f"{tabela}: {linhas} linhas")

consultas = carregar_consultas()
df = pd.read_csv("../data/processed/dataset_final_resumo_limpo.csv")  # para conferência


def mostrar(nome):
    """Imprime o SQL da consulta e devolve o resultado."""
    print(consultas[nome], end="\n\n")
    return consultar(nome)

eventos: 149 linhas
lutas: 1837 linhas
desempenho: 3674 linhas
desempenho_round: 8910 linhas


## O modelo

Tabelas e quantidade de linhas. A chave de `desempenho` é `(luta_id, lutador)`; cada luta tem exatamente dois registros.

In [2]:
with sqlite3.connect(ARQUIVO_BANCO) as conexao:
    tabelas = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY rowid", conexao)["name"]
    print(pd.DataFrame({t: [pd.read_sql_query(f"SELECT COUNT(*) FROM {t}", conexao).iloc[0, 0]] for t in tabelas}, index=["linhas"]).T)
    print("Lutas com número de lutadores diferente de 2:",
          pd.read_sql_query("SELECT COUNT(*) FROM (SELECT luta_id FROM desempenho GROUP BY luta_id HAVING COUNT(*) <> 2)", conexao).iloc[0, 0])

                  linhas
eventos              149
lutas               1837
desempenho          3674
desempenho_round    8910
Lutas com número de lutadores diferente de 2: 0


## 1. Como as lutas terminam

In [3]:
metodos = mostrar("metodos_vitoria")
metodos

-- Como as lutas terminam? Uma linha por luta, então não há contagem dobrada.
SELECT
    metodo,
    COUNT(*)                                            AS lutas,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)  AS pct
FROM lutas
GROUP BY metodo
ORDER BY lutas DESC;



,metodo,lutas,pct
0,Decision - Unanimous,731,39.8
1,KO/TKO,588,32.0
2,Submission,313,17.0
3,Decision - Split,158,8.6
4,Decision - Majority,15,0.8
5,Could Not Continue,13,0.7
6,TKO - Doctor's Stoppage,9,0.5
7,Overturned,7,0.4
8,DQ,3,0.2


In [4]:
# Conferência: mesma contagem do gráfico metodos_vitoria.png (uma linha por luta)
esperado = df.drop_duplicates("Fight_URL")["Method"].value_counts()
assert metodos.set_index("metodo")["lutas"].to_dict() == esperado.to_dict()
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem


## 2. O que separa vencedores de perdedores

In [5]:
vp = mostrar("vencedores_perdedores")
vp

-- O que separa quem vence de quem perde? Médias por luta de cada lado.
WITH medias AS (
    SELECT
        resultado,
        AVG(sig_acertados)            AS golpes_sig,
        AVG(quedas)                   AS quedas,
        AVG(controle_seg) / 60.0      AS controle_min,
        AVG(tent_finalizacao)         AS tent_finalizacao,
        AVG(kd)                       AS knockdowns
    FROM desempenho
    WHERE resultado IN ('W', 'L')
    GROUP BY resultado
)
SELECT
    ROUND(v.golpes_sig, 1)                         AS golpes_sig_v,
    ROUND(p.golpes_sig, 1)                         AS golpes_sig_p,
    ROUND(v.golpes_sig / p.golpes_sig, 1)          AS golpes_sig_razao,
    ROUND(v.quedas / p.quedas, 1)                  AS quedas_razao,
    ROUND(v.controle_min / p.controle_min, 1)      AS controle_razao,
    ROUND(v.tent_finalizacao / p.tent_finalizacao, 1) AS finalizacao_razao,
    ROUND(v.knockdowns / p.knockdowns, 1)          AS knockdowns_razao
FROM medias v
CROSS JOIN medias p


,golpes_sig_v,golpes_sig_p,golpes_sig_razao,quedas_razao,controle_razao,finalizacao_razao,knockdowns_razao
0,52.0,35.4,1.5,2.1,2.1,2.6,8.4


In [6]:
medias = df[df["Resultado"].isin(["W", "L"])].groupby("Resultado")[["Sig_str_landed", "KD", "Td_landed"]].mean()
assert round(medias.loc["W", "KD"] / medias.loc["L", "KD"], 1) == vp.loc[0, "knockdowns_razao"]
assert round(medias.loc["W", "Td_landed"] / medias.loc["L", "Td_landed"], 1) == vp.loc[0, "quedas_razao"]
assert round(medias.loc["W", "Sig_str_landed"], 1) == vp.loc[0, "golpes_sig_v"]
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem


## 3. Taxa de vitória (mín. 6 lutas)

`RANK()` deixa explícitos os empates: vários invictos dividem a 1ª posição.

In [7]:
taxa = mostrar("taxa_vitoria")
taxa

-- Top 10 por taxa de vitória, com amostra mínima de 6 lutas (NC não entra na conta).
-- RANK() mostra os empates: vários invictos dividem a 1ª posição.
SELECT
    RANK() OVER (ORDER BY 1.0 * SUM(resultado = 'W') / COUNT(*) DESC) AS posicao,
    lutador,
    SUM(resultado = 'W')                                  AS vitorias,
    SUM(resultado = 'L')                                  AS derrotas,
    SUM(resultado = 'D')                                  AS empates,
    ROUND(100.0 * SUM(resultado = 'W') / COUNT(*), 1)     AS taxa_pct
FROM desempenho
WHERE resultado IN ('W', 'L', 'D')
GROUP BY lutador
HAVING COUNT(*) >= 6
ORDER BY taxa_pct DESC, vitorias DESC
LIMIT 10;



,posicao,lutador,vitorias,derrotas,empates,taxa_pct
0,1,Carlos Ulberg,7,0,0,100.0
1,1,Ailin Perez,6,0,0,100.0
2,1,Farid Basharat,6,0,0,100.0
3,1,Natalia Silva,6,0,0,100.0
4,1,Navajo Stirling,6,0,0,100.0
5,1,Quillan Salkilld,6,0,0,100.0
6,7,Joshua Van,11,1,0,91.7
7,8,Asu Almabayev,7,1,0,87.5
8,8,Carlos Prates,7,1,0,87.5
9,8,Denise Gomes,7,1,0,87.5


In [8]:
cartel = pd.crosstab(df["Fighter"], df["Resultado"]).reindex(columns=["W", "L", "D"], fill_value=0)
cartel["lutas"] = cartel.sum(axis=1)
cartel = cartel[cartel["lutas"] >= 6]
cartel["taxa"] = cartel["W"] / cartel["lutas"] * 100
esperado = cartel.sort_values(["taxa", "W"], ascending=False).head(10)
assert list(taxa["lutador"]) == list(esperado.index)
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem


## 4. Precisão de quedas (mín. 15 tentativas e 3 lutas)

In [9]:
quedas = mostrar("precisao_quedas")
quedas

-- Top 10 por precisão de quedas: pelo menos 15 tentativas E 3 lutas no período.
SELECT
    lutador,
    SUM(quedas)                                           AS conseguidas,
    SUM(quedas_tentadas)                                  AS tentadas,
    COUNT(*)                                              AS lutas,
    ROUND(100.0 * SUM(quedas) / SUM(quedas_tentadas), 1)  AS precisao_pct
FROM desempenho
GROUP BY lutador
HAVING SUM(quedas_tentadas) >= 15 AND COUNT(*) >= 3
ORDER BY precisao_pct DESC, conseguidas DESC
LIMIT 10;



,lutador,conseguidas,tentadas,lutas,precisao_pct
0,Mason Jones,11,15,4,73.3
1,Andre Muniz,14,20,4,70.0
2,Fatima Kline,14,21,5,66.7
3,Joanderson Brito,14,21,8,66.7
4,Cory Sandhagen,10,15,5,66.7
5,Zhang Weili,13,20,4,65.0
6,Joaquin Buckley,12,19,8,63.2
7,Hamdy Abdelwahab,10,16,3,62.5
8,Lerone Murphy,10,16,6,62.5
9,Piera Rodriguez,18,29,5,62.1


In [10]:
g = df.groupby("Fighter").agg(c=("Td_landed", "sum"), t=("Td_attempted", "sum"), n=("Fight_URL", "nunique"))
g = g[(g["t"] >= 15) & (g["n"] >= 3)]
g["p"] = g["c"] / g["t"] * 100
esperado = g.sort_values(["p", "c"], ascending=False).head(10)
assert list(quedas["lutador"]) == list(esperado.index)
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem


## 5. Em que round terminam nocautes e finalizações

Percentual **dentro de cada método** com `SUM(COUNT(*)) OVER (PARTITION BY tipo)`.

In [11]:
rounds = mostrar("round_final_por_metodo")
rounds

-- Em que round terminam nocautes e finalizações? Percentual dentro de cada método
-- (função de janela particionada por método). Rounds 4 e 5 agrupados.
WITH finalizadas AS (
    SELECT
        CASE
            WHEN metodo LIKE '%KO/TKO%' OR metodo LIKE 'TKO - Doctor%' THEN 'Nocaute'
            WHEN metodo LIKE '%Submission%' THEN 'Finalização'
        END AS tipo,
        CASE WHEN round_final >= 4 THEN '4º ou 5º' ELSE round_final || 'º' END AS round_grupo
    FROM lutas
)
SELECT
    tipo,
    round_grupo,
    COUNT(*)                                                               AS lutas,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY tipo), 1)    AS pct_do_metodo
FROM finalizadas
WHERE tipo IS NOT NULL
GROUP BY tipo, round_grupo
ORDER BY tipo DESC, round_grupo;



,tipo,round_grupo,lutas,pct_do_metodo
0,Nocaute,1º,320,53.6
1,Nocaute,2º,176,29.5
2,Nocaute,3º,90,15.1
3,Nocaute,4º ou 5º,11,1.8
4,Finalização,1º,144,46.0
5,Finalização,2º,109,34.8
6,Finalização,3º,56,17.9
7,Finalização,4º ou 5º,4,1.3


In [12]:
assert rounds.query("tipo == 'Nocaute' and round_grupo == '1º'")["pct_do_metodo"].item() == 53.6
assert rounds.query("tipo == 'Finalização' and round_grupo == '1º'")["pct_do_metodo"].item() == 46.0
print("OK: mesmos percentuais do gráfico round_final_por_metodo.png")

OK: mesmos percentuais do gráfico round_final_por_metodo.png


## 6. De onde saem os golpes de vencedores e perdedores

In [13]:
origem = mostrar("origem_golpes")
origem

-- De onde saem os golpes de vencedores e perdedores (distância, clinch, chão)?
SELECT
    CASE resultado WHEN 'W' THEN 'Vencedores' ELSE 'Perdedores' END  AS grupo,
    ROUND(100.0 * SUM(distancia) / SUM(sig_acertados), 1)            AS distancia_pct,
    ROUND(100.0 * SUM(clinch)    / SUM(sig_acertados), 1)            AS clinch_pct,
    ROUND(100.0 * SUM(chao)      / SUM(sig_acertados), 1)            AS chao_pct
FROM desempenho
WHERE resultado IN ('W', 'L')
GROUP BY resultado
ORDER BY grupo DESC;



,grupo,distancia_pct,clinch_pct,chao_pct
0,Vencedores,77.3,9.5,13.2
1,Perdedores,86.8,9.7,3.5


In [14]:
wl = df[df["Resultado"].isin(["W", "L"])].groupby("Resultado")[["Ground_landed", "Sig_str_landed"]].sum()
chao = (wl["Ground_landed"] / wl["Sig_str_landed"] * 100).round(1)
assert origem.set_index("grupo").loc["Vencedores", "chao_pct"] == chao["W"]
assert origem.set_index("grupo").loc["Perdedores", "chao_pct"] == chao["L"]
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem


## 7. Quem mais arrisca golpes (por minuto de luta)

`JOIN` com `lutas` para usar a duração real de cada luta.

In [15]:
volume = mostrar("volume_por_minuto")
volume

-- Quem mais arrisca golpes? Tentativas por minuto de luta (JOIN com lutas para a duração),
-- lutadores com pelo menos 4 lutas.
SELECT
    d.lutador,
    COUNT(*)                                                       AS lutas,
    ROUND(SUM(d.sig_tentados) / (SUM(l.duracao_seg) / 60.0), 1)    AS tentados_por_min,
    ROUND(100.0 * SUM(d.sig_acertados) / SUM(d.sig_tentados), 1)   AS precisao_pct
FROM desempenho AS d
JOIN lutas AS l USING (luta_id)
GROUP BY d.lutador
HAVING COUNT(*) >= 4
ORDER BY tentados_por_min DESC
LIMIT 10;



,lutador,lutas,tentados_por_min,precisao_pct
0,Carlos Leal,4,18.6,47.0
1,Esteban Ribovics,7,18.6,43.1
2,Nasrat Haqparast,6,18.3,46.7
3,Carli Judice,5,18.1,48.1
4,Terrance McKinney,9,17.4,59.4
5,Djorden Santos,4,16.9,39.5
6,Mason Jones,4,16.6,43.1
7,Josh Hokit,4,16.1,63.8
8,Diana Belbita,4,15.7,41.3
9,Davey Grant,5,15.3,46.9


## 8. Árbitros

O árbitro não escolhe o método: a taxa reflete as lutas que cada um recebe.

In [16]:
arbitros = mostrar("arbitros")
arbitros

-- Os 10 árbitros com mais lutas e a % delas encerrada antes da decisão.
SELECT
    arbitro,
    COUNT(*)                                                                    AS lutas,
    ROUND(100.0 * SUM(metodo LIKE '%KO/TKO%' OR metodo LIKE 'TKO - Doctor%'
                      OR metodo LIKE '%Submission%') / COUNT(*), 1)             AS pct_antes_da_decisao
FROM lutas
GROUP BY arbitro
ORDER BY lutas DESC
LIMIT 10;



,arbitro,lutas,pct_antes_da_decisao
0,Herb Dean,212,55.2
1,Jason Herzog,197,47.2
2,Marc Goddard,194,51.0
3,Chris Tognoni,167,44.9
4,Mark Smith,159,51.6
5,Mike Beltran,117,53.0
6,Keith Peterson,113,39.8
7,Kerry Hatley,90,55.6
8,Dan Miragliotta,76,46.1
9,Rich Mitchell,36,41.7


## 9. Golpes por round

Tende a subir por viés de sobrevivência: só lutas longas chegam aos rounds finais.

In [17]:
mostrar("golpes_por_round")

-- Média de golpes significativos acertados por round (tende a subir por viés de sobrevivência:
-- só lutas longas chegam aos rounds finais).
SELECT
    round,
    COUNT(*)                     AS registros,
    ROUND(AVG(sig_acertados), 1) AS media_golpes_sig
FROM desempenho_round
GROUP BY round
ORDER BY round;



,round,registros,media_golpes_sig
0,1,3674,16.2
1,2,2724,18.9
2,3,2138,19.7
3,4,196,19.6
4,5,178,20.5


## 10. Consultas que só aparecem aqui

Perguntas que seriam trabalhosas em Pandas e ficam naturais com funções de janela.

### Destaque de cada evento

`ROW_NUMBER() OVER (PARTITION BY evento)` escolhe quem mais acertou golpes significativos numa luta, em cada evento.

In [18]:
mostrar("destaque_por_evento")

-- Em cada evento, quem acertou mais golpes significativos numa única luta?
-- ROW_NUMBER() por evento escolhe um só por evento; mostra os 10 maiores desses recordes.
WITH ranqueado AS (
    SELECT
        e.nome        AS evento,
        d.lutador,
        d.sig_acertados,
        ROW_NUMBER() OVER (PARTITION BY l.evento_id ORDER BY d.sig_acertados DESC, d.lutador) AS n
    FROM desempenho AS d
    JOIN lutas   AS l USING (luta_id)
    JOIN eventos AS e USING (evento_id)
)
SELECT evento, lutador, sig_acertados
FROM ranqueado
WHERE n = 1
ORDER BY sig_acertados DESC
LIMIT 10;



,evento,lutador,sig_acertados
0,UFC Fight Night: Du Plessis vs. Usman,Tommy McMillen,252
1,UFC Fight Night: Vettori vs. Cannonier,Jared Cannonier,241
2,UFC 299: O'Malley vs. Vera 2,Sean O'Malley,230
3,UFC Fight Night: Barboza vs. Murphy,Lerone Murphy,220
4,UFC 317: Topuria vs. Oliveira,Brandon Royval,215
5,UFC 296: Edwards vs. Covington,Karol Rosa,204
6,UFC 318: Holloway vs. Poirier 3,Max Holloway,198
7,UFC 300: Pereira vs. Hill,King Green,186
8,UFC 302: Makhachev vs. Poirier,Sean Strickland,182
9,UFC 331: Van vs. Pantoja 2,Joshua Van,181


### Maior sequência de vitórias

Problema clássico de *gaps and islands*: dentro de uma sequência de resultados iguais, a diferença entre o número da luta na carreira e o número da luta entre as de mesmo resultado é constante — e vira o identificador da sequência.

In [19]:
mostrar("sequencia_vitorias")

-- Maior sequência de vitórias seguidas de cada lutador no período ("gaps and islands"):
-- a diferença entre dois ROW_NUMBER() é constante dentro de uma sequência de mesmo resultado.
-- A ordem das lutas vem do evento: evento_id segue a ordem da coleta (1 = mais recente).
WITH ordenado AS (
    SELECT
        d.lutador,
        d.resultado,
        ROW_NUMBER() OVER (PARTITION BY d.lutador ORDER BY l.evento_id DESC)
          - ROW_NUMBER() OVER (PARTITION BY d.lutador, d.resultado ORDER BY l.evento_id DESC) AS grupo
    FROM desempenho AS d
    JOIN lutas AS l USING (luta_id)
),
sequencias AS (
    SELECT lutador, COUNT(*) AS vitorias_seguidas
    FROM ordenado
    WHERE resultado = 'W'
    GROUP BY lutador, grupo
)
SELECT lutador, MAX(vitorias_seguidas) AS maior_sequencia
FROM sequencias
GROUP BY lutador
ORDER BY maior_sequencia DESC, lutador
LIMIT 10;



,lutador,maior_sequencia
0,Joshua Van,8
1,Carlos Ulberg,7
2,Ailin Perez,6
3,Farid Basharat,6
4,Joaquin Buckley,6
5,Melquizael Costa,6
6,Natalia Silva,6
7,Navajo Stirling,6
8,Quillan Salkilld,6
9,Steve Garcia,6
